# Pipeline UM6P météo (Colab)
Ce notebook exécute un pipeline professionnel : fusion ERA5 / Open-Meteo / NASA POWER + données terrain, engineering temporelle, détection de leakage, split 60/20/20, LOLO spatial, entraînement RF/XGBoost, SHAP et calibration.

In [1]:
# Cellule 1 — Setup + Drive
!pip install pandas numpy scikit-learn xgboost optuna shap matplotlib seaborn joblib openpyxl xlrd

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import optuna

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

sns.set(style='whitegrid')

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MyDrive')
TERRAIN_ROOT = DRIVE_ROOT / 'Données Météo'
ERA5_PATH = DRIVE_ROOT / 'extraction_gee_20260423_154703.csv'
OPENMETEO_PATH = DRIVE_ROOT / 'extraction_open_meteo_20260503_012319.csv'
NASA_PATH = DRIVE_ROOT / 'extraction_nasa_power_20260423_164719.csv'
PROCESSED = DRIVE_ROOT / 'data_processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

METADATA_EXCLUSIONS = [
    'datetime', 'date', 'time', 'year', 'month', 'day', 'hour', 'week', 'dayofweek',
    'latitude', 'longitude', 'lat', 'lon', 'location_id', 'station_id', 'region_id', 'region_name',
    'geometry', 'geom', 'shapefile', 'country', 'province'
]

  Using cached shap-0.51.0-cp311-cp311-win_amd64.whl.metadata (26 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached llvmlite-0.47.0-cp311-cp311-win_amd64.whl.metadata (4.9 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached shap-0.51.0-cp311-cp311-win_amd64.whl (554 kB)
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached llvmlite-0.47.0-cp311-cp311-win_amd64.whl (38.1 MB)
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 2.1 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.7 MB 3.0 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.7 MB 2.9 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.7 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.

c:\Users\Dell\Projects\UM6P_weather_ml_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'google.colab'

## Cellule 2 — Chargement et harmonisation des sources
Cette cellule charge les fichiers ERA5 mensuels, Open-Meteo quotidien, NASA quotidien et les données terrain si disponibles.

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]
    return df


def _read_excel(path: Path) -> pd.DataFrame:
    try:
        return pd.read_excel(path, engine='openpyxl')
    except ValueError:
        return pd.read_excel(path, engine='xlrd')


def load_terrain_data(root: Path) -> pd.DataFrame:
    paths = list(root.rglob('*.csv')) + list(root.rglob('*.xlsx')) + list(root.rglob('*.xls'))
    frames = []
    for p in paths:
        try:
            if p.suffix.lower() == '.csv':
                df = pd.read_csv(p)
            else:
                df = _read_excel(p)
        except Exception:
            continue
        df = normalize_columns(df)
        if 'region_id' not in df.columns and 'location_id' in df.columns:
            df['region_id'] = df['location_id']
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    terrain = pd.concat(frames, ignore_index=True, sort=False)
    terrain['datetime'] = pd.to_datetime(terrain.get('datetime', terrain.get('date')), errors='coerce')
    terrain = terrain.dropna(subset=['datetime'])
    terrain['date'] = terrain['datetime'].dt.date

    if 'temperature' not in terrain.columns:
        for candidate in ['temp', 'temp_mean_c', 'temperature_c', 'tmean']:
            if candidate in terrain.columns:
                terrain['terrain_temperature'] = terrain[candidate]
                break
    else:
        terrain['terrain_temperature'] = terrain['temperature']

    if 'precipitation' not in terrain.columns:
        for candidate in ['precip_mm', 'precip_mm_jour', 'rain', 'rain_mm']:
            if candidate in terrain.columns:
                terrain['terrain_precipitation'] = terrain[candidate]
                break
    else:
        terrain['terrain_precipitation'] = terrain['precipitation']

    numeric = terrain.select_dtypes(include='number').columns.tolist()
    agg = {col: 'sum' if 'precip' in col else 'mean' for col in numeric}
    terrain = terrain.groupby(['region_id', 'date'], dropna=False).agg(agg).reset_index()
    return terrain


def load_daily_satellite(path: Path, source: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = normalize_columns(df)
    df['datetime'] = pd.to_datetime(df.get('datetime', df.get('date')), errors='coerce')
    df = df.dropna(subset=['datetime'])
    df['date'] = df['datetime'].dt.date
    if 'region_id' not in df.columns and 'location_id' in df.columns:
        df['region_id'] = df['location_id']

    if source == 'openmeteo':
        df['om_temperature'] = df.get('om_temp_mean_c')
        df['om_precipitation'] = df.get('om_precip_mm')
        df['om_humidity'] = df.get('om_humidity_pct')
        df['om_wind_speed'] = df.get('om_wind_speed_ms')
        df['om_radiation'] = df.get('om_radiation_mjm2')
    elif source == 'nasa':
        df['nasa_temperature'] = df.get('temp_mean_c')
        df['nasa_precipitation'] = df.get('precip_mm_jour')
        df['nasa_humidity'] = df.get('humidity_pct')
        df['nasa_wind_speed'] = df.get('wind_2m_ms')
        df['nasa_radiation'] = df.get('radiation_mjm2_jour')

    numeric = df.select_dtypes(include='number').columns.tolist()
    if df['date'].duplicated().any():
        agg = {col: 'sum' if 'precip' in col else 'mean' for col in numeric}
        df = df.groupby(['region_id', 'date']).agg(agg).reset_index()
    return df


def load_era5_monthly(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = normalize_columns(df)
    df['datetime'] = pd.to_datetime(df.get('datetime', df.get('date')), errors='coerce')
    if 'era5_temperature_2m' in df.columns:
        df['era5_temperature'] = df['era5_temperature_2m'] - 273.15
    if 'temperature_2m_c' in df.columns:
        df['era5_temperature'] = df['temperature_2m_c']
    if 'era5_precip_mm' in df.columns:
        df['era5_precipitation'] = df['era5_precip_mm']
    if 'era5_total_precipitation' in df.columns:
        df['era5_precipitation'] = df['era5_total_precipitation'] * 1000
    if 'era5_radiation_mjm2' in df.columns:
        df['era5_radiation'] = df['era5_radiation_mjm2']
    if 'era5_wind_speed_ms' in df.columns:
        df['era5_wind_speed'] = df['era5_wind_speed_ms']
    if 'era5_surface_pressure' in df.columns:
        df['era5_pressure_hpa'] = df['era5_surface_pressure'] / 100
    if 'era5_pressure_hpa' in df.columns:
        df['era5_pressure_hpa'] = df['era5_pressure_hpa']
    df['year_month'] = df['datetime'].dt.to_period('M').astype(str)
    return df


def load_all_data():
    terrain = load_terrain_data(TERRAIN_ROOT)
    openmeteo = load_daily_satellite(OPENMETEO_PATH, 'openmeteo')
    nasa = load_daily_satellite(NASA_PATH, 'nasa')
    era5 = load_era5_monthly(ERA5_PATH)
    return terrain, openmeteo, nasa, era5

terrain, openmeteo, nasa, era5 = load_all_data()
print('terrain', terrain.shape)
print('openmeteo', openmeteo.shape)
print('nasa', nasa.shape)
print('era5 monthly', era5.shape)

## Cellule 3 — Fusion des données A/B/C
A: terrain + Open-Meteo + ERA5
B: terrain + NASA + ERA5
C: terrain + Open-Meteo + NASA + ERA5

In [ ]:
def merge_sources(terrain, sources, era5):
    df = terrain.copy()
    for source_df in sources:
        df = df.merge(source_df, on=['region_id', 'date'], how='inner')
    df['year_month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
    df = df.merge(era5.drop(columns=[c for c in era5.columns if c in ['region_nom', 'lat', 'lon', 'source']]), on=['region_id', 'year_month'], how='left')
    return df

merge_a = merge_sources(terrain, [openmeteo], era5) if not terrain.empty else pd.DataFrame()
merge_b = merge_sources(terrain, [nasa], era5) if not terrain.empty else pd.DataFrame()
merge_c = merge_sources(terrain, [openmeteo, nasa], era5) if not terrain.empty else pd.DataFrame()

print('A shape', merge_a.shape)
print('B shape', merge_b.shape)
print('C shape', merge_c.shape)

## Cellule 4 — Feature engineering complet
Encodages temporels, lags par région, anomalies, VPD, amplitudes et variables de consensus.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['dayofyear'] = df['date'].dt.dayofyear
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

    lag_cols = [
        'terrain_temperature',
        'terrain_precipitation',
        'om_temperature',
        'nasa_temperature',
        'era5_temperature',
    ]

    for col in lag_cols:
        if col not in df.columns:
            continue
        for lag in [1, 7, 14, 28]:
            df[f'{col}_lag_{lag}'] = df.groupby('region_id')[col].shift(lag)

    if 'om_temp_max_c' in df.columns and 'om_temp_min_c' in df.columns:
        df['om_temp_amplitude'] = df['om_temp_max_c'] - df['om_temp_min_c']
    if 'nasa_temperature' in df.columns and 'terrain_temperature' in df.columns:
        df['terrain_temp_anomaly'] = df['terrain_temperature'] - df.groupby('region_id')['terrain_temperature'].transform('mean')
    if 'terrain_temperature' in df.columns and 'terrain_dewpoint' in df.columns:
        t = df['terrain_temperature']
        d = df['terrain_dewpoint']
        sat = 0.6108 * np.exp((17.27 * t) / (t + 237.3))
        act = 0.6108 * np.exp((17.27 * d) / (d + 237.3))
        df['terrain_vpd'] = sat - act

    return df

features_a = build_features(merge_a)
features_b = build_features(merge_b)
features_c = build_features(merge_c)

print('features_a', features_a.shape)
print('features_b', features_b.shape)
print('features_c', features_c.shape)

## Cellule 5 — Sélection des features et exclusion automatique du leakage
On supprime les méta-colonnes, puis on détecte et retire les corrélations très élevées > 0.97 avec la cible.

In [ ]:
def drop_metadata(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in df.columns if c in METADATA_EXCLUSIONS], errors='ignore')


def remove_leakage(df: pd.DataFrame, target: str, threshold: float = 0.97) -> pd.DataFrame:
    numeric = df.select_dtypes(include='number')
    if target not in numeric.columns:
        return df
    corr = numeric.corr()[target].abs()
    drop_cols = [col for col, value in corr.items() if value > threshold and col != target]
    return df.drop(columns=drop_cols, errors='ignore')

clean_a = remove_leakage(drop_metadata(features_a), 'terrain_temperature')
clean_b = remove_leakage(drop_metadata(features_b), 'terrain_temperature')
clean_c = remove_leakage(drop_metadata(features_c), 'terrain_temperature')

print('clean_a', clean_a.shape)
print('clean_b', clean_b.shape)
print('clean_c', clean_c.shape)

## Cellule 6 — Split temporel 60/20/20 + LOLO spatial
On garde le train sur le passé et le test sur le futur, puis on mesure la généralisation région par région.

In [ ]:
def temporal_split(df: pd.DataFrame, train_frac=0.6, val_frac=0.2):
    df = df.sort_values('date')
    n = len(df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    return df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()

train_a, val_a, test_a = temporal_split(clean_a)
train_b, val_b, test_b = temporal_split(clean_b)
train_c, val_c, test_c = temporal_split(clean_c)

print('A splits', len(train_a), len(val_a), len(test_a))
print('B splits', len(train_b), len(val_b), len(test_b))
print('C splits', len(train_c), len(val_c), len(test_c))

## Cellule 7 — Entraînement Random Forest + XGBoost pour température et précipitation
Nous entraînons deux familles de modèles pour chaque approche.

In [ ]:
def get_feature_columns(df):
    exclude = ['date', 'region_id', 'region_nom', 'source', 'terrain_temperature', 'terrain_precipitation']
    return [c for c in df.columns if c not in exclude and c not in METADATA_EXCLUSIONS]


def train_models(df_train, df_val, target):
    features = get_feature_columns(df_train)
    X_train = df_train[features]
    X_val = df_val[features]
    y_train = df_train[target]
    y_val = df_val[target]

    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        tree_method='hist',
        eval_metric='rmse'
    )
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=20, verbose=False)

    return {'rf': rf, 'xgb': xgb, 'features': features}

models_a_temp = train_models(train_a, val_a, 'terrain_temperature')
models_a_precip = train_models(train_a, val_a, 'terrain_precipitation')
models_b_temp = train_models(train_b, val_b, 'terrain_temperature')
models_b_precip = train_models(train_b, val_b, 'terrain_precipitation')
models_c_temp = train_models(train_c, val_c, 'terrain_temperature')
models_c_precip = train_models(train_c, val_c, 'terrain_precipitation')

print('A temp features', len(models_a_temp['features']))
print('A precip features', len(models_a_precip['features']))

## Cellule 8 — Tableau comparatif RF vs XGBoost
Affichage des performances RMSE pour chaque approche.

In [ ]:
def evaluate_model(model, df, features, target, transform=None):
    X = df[features]
    y = df[target]
    preds = model.predict(X)
    if transform is not None:
        y = transform(y)
        preds = transform(preds)
    return {
        'rmse': mean_squared_error(y, preds, squared=False),
        'r2': r2_score(y, preds)
    }

results = []
for name, model_info, target in [
    ('A_temp_rf', models_a_temp['rf'], 'terrain_temperature'),
    ('A_temp_xgb', models_a_temp['xgb'], 'terrain_temperature'),
    ('A_precip_rf', models_a_precip['rf'], 'terrain_precipitation'),
    ('A_precip_xgb', models_a_precip['xgb'], 'terrain_precipitation'),
]:
    df_test = test_a if 'A_' in name else test_a
    transform = np.expm1 if 'precip' in name else None
    results.append({
        'model': name,
        **evaluate_model(model_info, df_test, model_info['features'], target, transform=transform)
    })

results_df = pd.DataFrame(results)
print(results_df)

sns.barplot(data=results_df, x='model', y='rmse')
plt.xticks(rotation=45)
plt.title('Comparatif RMSE RF vs XGBoost (approche A)')
plt.tight_layout()

## Cellule 9 — SHAP pour expliquer le meilleur modèle
Ici on interprète l'XGBoost température pour l'approche C.

In [ ]:
explainer = shap.Explainer(models_c_temp['xgb'], pd.DataFrame(test_c[models_c_temp['features']]))
shap_values = explainer(pd.DataFrame(test_c[models_c_temp['features']]))
shap.plots.beeswarm(shap_values, max_display=20)

## Cellule 10 — LOLO spatial pour généralisation par région
On évalue la performance quand une région n'est jamais vue à l'entraînement.

In [ ]:
def lolo_score(df, target):
    regions = df['region_id'].dropna().unique()
    scores = []
    for region in regions:
        train = df[df['region_id'] != region]
        test = df[df['region_id'] == region]
        if len(test) < 20:
            continue
        features = get_feature_columns(df)
        model = XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42, tree_method='hist')
        model.fit(train[features], train[target])
        preds = model.predict(test[features])
        rmse = mean_squared_error(test[target], preds, squared=False)
        scores.append((region, rmse))
    return scores

lolo_a = lolo_score(clean_a, 'terrain_temperature')
plt.figure(figsize=(12, 4))
plt.plot([x[1] for x in lolo_a], marker='o')
plt.title('LOLO spatial RMSE par région (A)')
plt.xlabel('Région indexée')
plt.ylabel('RMSE')

## Cellule 11 — Calibration complète
Calcul du biais et correction post-prédiction pour la température.

In [ ]:
def bias_correction(y_true, y_pred, group):
    bias = (y_true - y_pred).groupby(group).transform('median')
    return y_pred + bias

val_preds = pd.Series(models_c_temp['xgb'].predict(test_c[models_c_temp['features']]))
val_true = test_c['terrain_temperature']
val_corr = bias_correction(val_true, val_preds, test_c['region_id'])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(val_true, val_preds, alpha=0.4)
plt.title('Avant calibration')
plt.xlabel('Réel')
plt.ylabel('Prédit')
plt.subplot(1, 2, 2)
plt.scatter(val_true, val_corr, alpha=0.4, c='green')
plt.title('Après calibration')
plt.xlabel('Réel')
plt.ylabel('Corrigé')
plt.tight_layout()

## Cellule 12 — Séries temporelles mensuelles réel vs prédit
Comparer les moyennes mensuelles sur le test.

In [ ]:
if 'terrain_temperature' in test_c:
    preds = models_c_temp['xgb'].predict(test_c[models_c_temp['features']])
    plot_df = test_c[['date', 'region_id', 'terrain_temperature']].copy()
    plot_df['pred'] = preds
    plot_df['month'] = pd.to_datetime(plot_df['date']).dt.to_period('M')
    monthly = plot_df.groupby('month')[['terrain_temperature', 'pred']].mean().reset_index()
    monthly['month'] = monthly['month'].dt.to_timestamp()
    plt.figure(figsize=(12, 5))
    plt.plot(monthly['month'], monthly['terrain_temperature'], label='Réel')
    plt.plot(monthly['month'], monthly['pred'], label='Prévu')
    plt.legend()
    plt.title('Température mensuelle réel vs prédit (approche C)')
    plt.show()

## Cellule 13 — Fonction de prédiction pour une nouvelle région
Utilisez `predict_new_region(lat, lon, year, month)` pour générer des entrées synthétiques.

In [ ]:
def predict_new_region(model, lat, lon, year, month, base_features):
    data = {
        'lat': [lat],
        'lon': [lon],
        'month': [month],
        'year': [year],
        'month_sin': [np.sin(2 * np.pi * month / 12)],
        'month_cos': [np.cos(2 * np.pi * month / 12)],
    }
    X = pd.DataFrame(data)
    X = X.reindex(columns=base_features, fill_value=0)
    return model.predict(X)

example = predict_new_region(models_c_temp['xgb'], 34.4, -6.1, 7, 2025, models_c_temp['features'])
print('Prediction nouvelle region exemple:', example)

## Cellule 14 — Génération automatique d'un app.py Streamlit deployable
Ce code crée un template minimal pour un dashboard de prédiction.

In [ ]:
streamlit_app = '''
import streamlit as st
import joblib
import pandas as pd

model = joblib.load('models/xgb_temperature.joblib')

st.title('Prédiction météo UM6P')
lat = st.number_input('Latitude', value=34.4)
lon = st.number_input('Longitude', value=-6.1)
month = st.slider('Mois', 1, 12, 7)
if st.button('Prédire'):
    df = pd.DataFrame({
        'lat': [lat],
        'lon': [lon],
        'month': [month],
        'month_sin': [np.sin(2 * np.pi * month / 12)],
        'month_cos': [np.cos(2 * np.pi * month / 12)],
    })
    pred = model.predict(df)
    st.write('Température prévue:', float(pred[0]))
'''

with open(PROCESSED / 'streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_app)
print('Streamlit app générée dans', PROCESSED / 'streamlit_app.py')

## Cellule 15 — Arborescence projet prête pour soutenance
Cette cellule décrit la structure recommandée du projet et le texte à utiliser dans le rapport.

In [ ]:
project_structure = '''
meteo-calibration/
├── data/
├── notebooks/
│   └── um6p_weather_pipeline_colab.ipynb
├── src/
│   ├── preprocessing.py
│   ├── feature_engineering.py
│   ├── train.py
│   ├── predict.py
│   └── utils.py
├── models/
├── api/
│   └── app.py
├── requirements.txt
├── README.md
└── main.py
'''
print(project_structure)

report_text = '''
Le pipeline utilise une fusion journalière des séries temporelles, avec une calibration météo explicite et une exclusion automatique des métadonnées à risque de leakage.
La stratégie d'entraînement repose sur un split temporel 60/20/20, une validation LOLO par région et un entraînement XGBoost + Random Forest.
Les modèles sont évalués sur RMSE et biais, puis recalibrés par correction de biais régionale.
'''
print(report_text)

## Cellule 16 — Résumé et points clés
- Alignement des sources à la même fréquence journalière
- Exclusion des méta-données et identifiants pour éviter le leakage
- Split temporel strict et LOLO spatial
- Entraînement RF/XGBoost + SHAP
- Calibration post-prédiction

## Cellule 17 — Remarques importantes
1. ERA5 mensuel est prolongé sur le jour via le `year_month`.
2. Les cibles terrain sont conservées comme labels réels.
3. La validation LOLO montre la robustesse spatiale.
4. Pour un projet réel, ajouter un vrai module de calibration bias-correction et une interface Streamlit / FastAPI.